In [17]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_groq import ChatGroq
from dotenv import load_dotenv

In [18]:
load_dotenv()

True

In [19]:
llm = ChatGroq(model='openai/gpt-oss-120b')

In [9]:
parser = JsonOutputParser()

In [12]:
prompt = PromptTemplate(
    template='Provide me with Age, Name, City of a fictional person and {specific_format}',
    input_variables=[],
    partial_variables={'specific_format': parser.get_format_instructions()}
)

In [11]:
prompt

PromptTemplate(input_variables=[], input_types={}, partial_variables={'specific_format': 'Return a JSON object.'}, template='Provide me with Age, Name, City of a fition person and {specific_format}')

In [13]:
final_promt = prompt.format()
final_promt

'Provide me with Age, Name, City of a fictional person and Return a JSON object.'

In [16]:
res = llm.invoke(final_promt)
print(res.content)

Here is a JSON object with a fictional person's details:

```json
{
  "name": "Lena Whitaker",
  "age": 34,
  "city": "Portland"
}
```


In [24]:
parser.parse(res.content)

{'name': 'Lena Whitaker', 'age': 34, 'city': 'Portland'}

## same with chain

In [27]:
chain = prompt | llm | parser

In [28]:
chain_result = chain.invoke({})
print(chain_result)

{'Name': 'Lena Hartley', 'Age': 34, 'City': 'Cedarbrook'}


In [29]:
chain_result['Name']

'Lena Hartley'

In [6]:
## Pydantic O/P parser

from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import PromptTemplate

In [3]:
class Data(BaseModel):
    name: str = Field(description='name of the user')
    age: int = Field(gt=21, description='age of the user')
    city: str = Field(description='city will be from state of Uttar Pradesh')
    state: str =Field(description='the state has to be Uttar Pradesh')

In [4]:
parser = PydanticOutputParser(pydantic_object=Data)
parser

PydanticOutputParser(pydantic_object=<class '__main__.Data'>)

In [8]:
prompt = PromptTemplate(
    template='Provide the name, age, city and state of a fictional person from {nationality} in {format}',
    input_variables=['nationality'],
    partial_variables={'format': parser.get_format_instructions()}
)

In [12]:
res_prmpt = prompt.invoke({'nationality': 'India'})

In [13]:
print(res_prmpt)

text='Provide the name, age, city and state of a fictional person from India in The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "name of the user", "title": "Name", "type": "string"}, "age": {"description": "age of the user", "exclusiveMinimum": 21, "title": "Age", "type": "integer"}, "city": {"description": "city will be from state of Uttar Pradesh", "title": "City", "type": "string"}, "state": {"description": "the state has to be Uttar Pradesh", "title": "State", "type": "string"}}, "required": ["name", "age", "city", "state"]}\n```'


In [28]:
respo = llm.invoke(res_prmpt)

In [29]:
print(respo.content)

{
  "name": "Arjun Sharma",
  "age":  thirty-two,
  "city": "Lucknow",
  "state": "Uttar Pradesh"
}


In [37]:
chain = prompt | llm | parser

In [39]:
ress = chain.invoke({'nationality': 'India'})
ress

Data(name='Arjun Sharma', age=28, city='Lucknow', state='Uttar Pradesh')